# Optimizations vs baseline — Opus CLI

Each CLI optimization (`opt1-batch` … `opt6-stable-output`) is compared against the **baseline**:
config `1_hw-full-cli-sdk-skills-opus`, restricted to its plain `cli` / no-skills runs (same model, same
interface family, no skills). Every optimization is also Opus / CLI / no-skills, so this is an apples-to-apples
comparison of the optimization *only*.

For each optimization we average three metrics **across all tasks**, paired on the set of tasks that both the
optimization and the baseline ran (so a missing task never skews the mean):

- **local_time_s** — wall time spent locally (lower is better)
- **cost_usd** — dollar cost of the run (lower is better)
- **pass rate** — `asserts_passed / total_asserts` (higher is better)

Only `valid` runs are counted.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.csv")
for c in ["asserts_passed", "total_asserts", "local_time_s", "cost_usd"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["valid"] = df["valid"].astype(str).str.lower() == "true"
df["pass_rate"] = df["asserts_passed"] / df["total_asserts"]

BASELINE_CFG = "1_hw-full-cli-sdk-skills-opus"
# Optimizations in order, with short display labels.
OPTS = [
    ("5_hw-cli-opt1-batch-opus",          "opt1-batch"),
    ("6_hw-cli-opt2-session-reuse-opus",  "opt2-session-reuse"),
    ("7_hw-cli-opt3-compact-json-opus",   "opt3-compact-json"),
    ("8_hw-cli-opt4-idempotent-opus",     "opt4-idempotent"),
    ("9_hw-cli-opt5-quiet-opus",          "opt5-quiet"),
    ("10_hw-cli-opt6-stable-output-opus", "opt6-stable-output"),
]
METRICS = [
    ("local_time_s", "local time (s)", "min"),
    ("cost_usd",     "cost (USD)",     "min"),
    ("pass_rate",    "assert pass rate", "max"),
]

valid = df[df.valid].copy()
# Baseline = plain CLI, no skills.
base = valid[(valid.config == BASELINE_CFG) & (valid.interface == "cli") & (valid.skills == "none")]

def per_task_mean(sub, col):
    return sub.groupby("task")[col].mean()


## Paired comparison table

In [2]:
rows = []
for cfg, label in OPTS:
    opt = valid[valid.config == cfg]
    rec = {"optimization": label, "n_tasks": None}
    for col, _, _ in METRICS:
        b = per_task_mean(base, col)
        o = per_task_mean(opt, col)
        common = b.index.intersection(o.index)
        common = [t for t in common if pd.notna(b[t]) and pd.notna(o[t])]
        rec["n_tasks"] = len(common)
        bm, om = b[common].mean(), o[common].mean()
        rec[f"base_{col}"] = bm
        rec[f"opt_{col}"]  = om
        rec[f"delta_{col}"] = om - bm
        rec[f"pct_{col}"]   = (om - bm) / bm * 100 if bm else float("nan")
    rows.append(rec)

summary = pd.DataFrame(rows).set_index("optimization")
fmt = {}
for col, _, _ in METRICS:
    for p in ("base_", "opt_", "delta_"):
        fmt[p + col] = "{:.4f}" if col != "local_time_s" else "{:.1f}"
    fmt["pct_" + col] = "{:+.1f}%"
summary.style.format(fmt)


,n_tasks,base_local_time_s,opt_local_time_s,delta_local_time_s,pct_local_time_s,base_cost_usd,opt_cost_usd,delta_cost_usd,pct_cost_usd,base_pass_rate,opt_pass_rate,delta_pass_rate,pct_pass_rate
optimization,,,,,,,,,,,,,
opt1-batch,23,190.7,247.5,56.8,+29.8%,0.2742,0.3248,0.0507,+18.5%,1.0000,0.9855,-0.0145,-1.4%
opt2-session-reuse,24,196.9,213.7,16.7,+8.5%,0.2859,0.3102,0.0243,+8.5%,1.0000,0.9722,-0.0278,-2.8%
opt3-compact-json,23,193.9,222.6,28.8,+14.8%,0.2793,0.3055,0.0262,+9.4%,1.0000,0.9710,-0.0290,-2.9%
opt4-idempotent,25,214.4,246.1,31.7,+14.8%,0.3169,0.3637,0.0468,+14.8%,1.0000,1.0000,0.0000,+0.0%
opt5-quiet,24,196.9,222.4,25.4,+12.9%,0.2859,0.3001,0.0142,+5.0%,1.0000,0.9861,-0.0139,-1.4%
opt6-stable-output,23,194.6,205.2,10.6,+5.4%,0.2815,0.2959,0.0143,+5.1%,1.0000,0.9710,-0.0290,-2.9%


## Charts

Baseline (grey) plus one bar per optimization. Green = better than baseline, red = worse
(direction-aware: lower is better for time/cost, higher for pass rate).

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
labels = [lbl for _, lbl in OPTS]

for ax, (col, title, direction) in zip(axes, METRICS):
    base_val = summary[f"base_{col}"].iloc[0]  # baseline is per-opt common-set mean; show first as reference
    # Use the baseline mean over the full baseline task set for the reference bar.
    base_full = per_task_mean(base, col).mean()
    opt_vals = summary[f"opt_{col}"].values
    deltas = summary[f"delta_{col}"].values
    better = (deltas < 0) if direction == "min" else (deltas > 0)
    colors = ["#2e7d32" if b else "#c62828" for b in better]

    xs = range(len(labels) + 1)
    heights = [base_full] + list(opt_vals)
    bar_colors = ["#9e9e9e"] + colors
    bars = ax.bar(xs, heights, color=bar_colors)
    ax.axhline(base_full, ls="--", lw=1, color="#555")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(["baseline"] + labels, rotation=45, ha="right")
    ax.set_title(f"{title}  ({'lower' if direction=='min' else 'higher'} = better)")
    ax.set_ylabel(title)
    for b, h in zip(bars, heights):
        ax.annotate(f"{h:.3g}", (b.get_x() + b.get_width()/2, h),
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()


## Assertions: passed / total across the paired tasks

In [4]:
arows = []
for cfg, label in OPTS:
    opt = valid[valid.config == cfg]
    b_ok, b_tot = per_task_mean(base, "asserts_passed"), per_task_mean(base, "total_asserts")
    o_ok, o_tot = per_task_mean(opt,  "asserts_passed"), per_task_mean(opt,  "total_asserts")
    common = b_ok.index.intersection(o_ok.index)
    arows.append({
        "optimization": label,
        "n_tasks": len(common),
        "base_passed": b_ok[common].sum(), "base_total": b_tot[common].sum(),
        "opt_passed":  o_ok[common].sum(), "opt_total":  o_tot[common].sum(),
    })
adf = pd.DataFrame(arows).set_index("optimization")
adf["base_rate"] = adf.base_passed / adf.base_total
adf["opt_rate"]  = adf.opt_passed  / adf.opt_total
adf.style.format({"base_passed": "{:.0f}", "base_total": "{:.0f}",
                  "opt_passed": "{:.0f}", "opt_total": "{:.0f}",
                  "base_rate": "{:.1%}", "opt_rate": "{:.1%}"})


,n_tasks,base_passed,base_total,opt_passed,opt_total,base_rate,opt_rate
optimization,,,,,,,
opt1-batch,23,76,76,75,76,100.0%,98.7%
opt2-session-reuse,24,79,79,77,79,100.0%,97.5%
opt3-compact-json,23,73,73,71,73,100.0%,97.3%
opt4-idempotent,25,83,83,83,83,100.0%,100.0%
opt5-quiet,24,79,79,78,79,100.0%,98.7%
opt6-stable-output,23,73,73,71,73,100.0%,97.3%
